# Building a Small Language Model (SLM) from scratch

**Dataset:** TinyStories  |  **Tokenizer:** GPT-2 BPE (tiktoken)  |  **Framework:** PyTorch

### Roadmap
| Step | What we do | Status |
|------|-----------|--------|
| 1 | Load the TinyStories dataset | done |
| 2 | Tokenize the stories and save token IDs to `train.bin` / `validation.bin` | done |
| 3 | Build a training batch (inputs + targets) and the input embeddings | done |
| 4 | LayerNorm | next |
| 5 | Masked multi-head causal attention | next |
| 6 | Feed-forward network (MLP) | todo |
| 7 | Transformer block, full model, output head | todo |
| 8 | Loss function and training loop | todo |

## Step 1: Load the dataset

TinyStories is a collection of short, simple stories written for small language models.
It comes with two splits: `train` and `validation`.

In [13]:
from datasets import load_dataset

# Download (or load from cache) the TinyStories dataset from Hugging Face
dataset = load_dataset("roneneldan/TinyStories")

# The two splits we will use
train_data = dataset["train"]
val_data = dataset["validation"]

# Look at one story to confirm the data loaded correctly
print(train_data[1]["text"])

Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Beep liked how the leaves fall and wanted to play with them. Beep drove under the tree and watched the leaves fall on him. He laughed and beeped his horn.

Beep played with the falling leaves all day. When it was time to go home, Beep knew he needed more fuel. He went to the fuel place and got more healthy fuel. Now, Beep was ready to go fast and play again the next day. And Beep lived happily ever after.


## Step 2: Tokenization

A neural network cannot read text, only numbers. A **tokenizer** splits text into pieces
(tokens) and maps each piece to an integer ID.

We use the GPT-2 tokenizer (Byte-Pair Encoding) through `tiktoken`:
- Vocabulary size: **50,257** tokens
- Special token `<|endoftext|>` (ID `50256`), which we put at the end of every story so the model can tell where one story stops and the next begins.

In [14]:
import tiktoken

# Load the GPT-2 tokenizer
encoding = tiktoken.get_encoding("gpt2")

# Encode one story to see what tokenization looks like
sample_story = train_data[0]["text"]
sample_story_tokens = encoding.encode(sample_story)

print(sample_story_tokens)               # the list of token IDs
print(f"Number of tokens: {len(sample_story_tokens)}")

[3198, 1110, 11, 257, 1310, 2576, 3706, 20037, 1043, 257, 17598, 287, 607, 2119, 13, 1375, 2993, 340, 373, 2408, 284, 711, 351, 340, 780, 340, 373, 7786, 13, 20037, 2227, 284, 2648, 262, 17598, 351, 607, 1995, 11, 523, 673, 714, 34249, 257, 4936, 319, 607, 10147, 13, 198, 198, 43, 813, 1816, 284, 607, 1995, 290, 531, 11, 366, 29252, 11, 314, 1043, 428, 17598, 13, 1680, 345, 2648, 340, 351, 502, 290, 34249, 616, 10147, 1701, 2332, 1995, 13541, 290, 531, 11, 366, 5297, 11, 20037, 11, 356, 460, 2648, 262, 17598, 290, 4259, 534, 10147, 526, 198, 198, 41631, 11, 484, 4888, 262, 17598, 290, 384, 19103, 262, 4936, 319, 20037, 338, 10147, 13, 632, 373, 407, 2408, 329, 606, 780, 484, 547, 7373, 290, 5742, 1123, 584, 13, 2293, 484, 5201, 11, 20037, 26280, 607, 1995, 329, 7373, 262, 17598, 290, 18682, 607, 10147, 13, 1119, 1111, 2936, 3772, 780, 484, 550, 4888, 290, 3111, 1978, 13]
Number of tokens: 162


### From token IDs to embeddings

`tiktoken` is just a lookup table: text <-> integer IDs. The model needs each ID turned into a
**vector** (an *embedding*). That lookup table is a matrix of shape `(vocab_size, embedding_dim)`.
Row `i` of the matrix is the vector for token ID `i`.

Here we peek at GPT-2's pretrained embedding matrix to see what the vectors look like.

In [15]:
import torch
from transformers import AutoModel

# Tokenize a short phrase
sample_token_ids = encoding.encode("Hello, World")
print(sample_token_ids)                  # [15496, 11, 2159]

# Load pretrained GPT-2 (we only need its embedding table)
gpt2 = AutoModel.from_pretrained("gpt2")

# Look up the embedding vector for each token ID
# Result shape: (num_tokens, 768)
sample_embeddings = gpt2.get_input_embeddings()(torch.tensor(sample_token_ids))
print(sample_embeddings)

[15496, 11, 2159]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 285.30it/s]

tensor([[-0.0687, -0.1327,  0.0112,  ...,  0.0715, -0.0297, -0.0477],
        [ 0.0115, -0.0029,  0.0323,  ...,  0.0277, -0.0297, -0.0599],
        [-0.0907,  0.0207,  0.3005,  ..., -0.2265, -0.0941,  0.0388]],
       grad_fn=<EmbeddingBackward0>)


## Step 2b: Save all tokens to `train.bin` and `validation.bin`

Tokenizing the full dataset takes a while, so we do it **once** and save the result to disk.

Design choices:
- **One long 1-D stream of IDs.** All stories are concatenated, each ending with the `<|endoftext|>` token.
- **`uint16` storage.** GPT-2's vocab (50,257) fits in 16 bits (max 65,535), so each token takes only 2 bytes.
- **Processed in chunks of 2,048 stories** so we never hold the whole dataset in RAM.

In [ ]:
import numpy as np

eot_token = encoding.eot_token       # ID of <|endoftext|> (50256)
CHUNK_SIZE = 2048                    # number of stories processed at a time


def tokenize_split_to_bin(split_data, out_path, chunk_size=CHUNK_SIZE):
    """
    Tokenize every story in `split_data` and write the token IDs to `out_path`
    as a flat binary file of uint16 values.
    """
    total_stories = len(split_data)

    # "wb" = write binary (creates the file, or overwrites it if it already exists)
    with open(out_path, "wb") as f:
        for start in range(0, total_stories, chunk_size):
            end = min(start + chunk_size, total_stories)
            chunk_texts = split_data[start:end]["text"]   # list of stories

            # Flatten the chunk into one list of token IDs
            chunk_tokens = []
            for text in chunk_texts:
                # encode_ordinary ignores special tokens in the text, so it is faster and safer
                tokens = encoding.encode_ordinary(text)
                tokens.append(eot_token)                  # mark the end of the story
                chunk_tokens.extend(tokens)

            # Convert to a uint16 array and append its raw bytes to the file
            f.write(np.array(chunk_tokens, dtype=np.uint16).tobytes())

    print(f"Data written to {out_path} successfully.")


tokenize_split_to_bin(train_data, "train.bin")
tokenize_split_to_bin(val_data, "validation.bin")

### Sanity check: read the file back with memory mapping

`np.memmap` lets us treat a huge file on disk like a NumPy array **without loading it into RAM**.
Only the parts we index are actually read. We decode a slice back to text to check the file is not corrupted.

In [16]:
import numpy as np
# Open the training tokens as a read-only memory-mapped array
train_tokens = np.memmap("train.bin", dtype=np.uint16, mode="r")

print("Total training tokens:", len(train_tokens))   # roughly 474 million

# Decode a slice of 100 tokens back to text; it should read as normal English
print(encoding.decode(train_tokens[1100:1200]))

Total training tokens: 473992236
 because she lost her favorite toy, a triangle. She looked everywhere in her house but could not find it.

One sunny day, Lily went to the park to play. She saw a big puddle of water and thought her triangle might be there. She put her hand in the water to soak it and looked for her toy. She felt something at the bottom of the puddle.

Lily pulled it out and saw that it was her triangle! She was so happy that she


## Step 3: Build a training batch

A language model learns **next-token prediction**: given the tokens so far, predict the next one.

We pick random windows of `context_length` tokens from the data:
- **Input `x`**:  tokens `i ... i + context_length - 1`
- **Target `y`**: tokens `i + 1 ... i + context_length` (the same window **shifted by one**)

At every position, the target is simply the token that comes next:

```
x = [ A   B   C   D ]
y = [ B   C   D   E ]     # when the model sees A it should predict B, then B -> C, etc.
```

Note that a window can cross a story boundary. This is normal. The `<|endoftext|>` token
tells the model that a new story is starting.

In [17]:
import torch

def get_batch(split, batch_size, context_length):
    """
    Return a random batch of (inputs, targets) for next-token prediction.

    Both tensors have shape (batch_size, context_length) and dtype torch.long.
    """
    # Memory-map the right file (cheap: nothing is loaded into RAM yet)
    path = "train.bin" if split == "train" else "validation.bin"
    data = np.memmap(path, dtype=np.uint16, mode="r")

    # Random start positions. We need `context_length + 1` tokens per window
    # (inputs plus one extra token for the final target), so the highest valid
    # start index is len(data) - context_length - 1.
    starts = np.random.randint(0, len(data) - context_length - 1, size=batch_size)

    # Inputs: window of tokens starting at i
    # Targets: the same window shifted one token to the right
    x = np.stack([data[i : i + context_length].astype(np.int64) for i in starts])
    y = np.stack([data[i + 1 : i + 1 + context_length].astype(np.int64) for i in starts])

    return torch.from_numpy(x), torch.from_numpy(y)


# Small values for now so we can inspect the output easily
context_length = 4
batch_size = 4

input_batch, output_batch = get_batch("train", batch_size, context_length)

print("Input batch :", input_batch.shape)
print(input_batch)
print("Target batch:", output_batch.shape)
print(output_batch)

Input batch : torch.Size([4, 4])
tensor([[   11,  5045,  1820,  2067],
        [  262,  3234,    13,   198],
        [  366, 13787,   318,   407],
        [  465,   835,   510,    13]])
Target batch: torch.Size([4, 4])
tensor([[ 5045,  1820,  2067,   284],
        [ 3234,    13,   198,   198],
        [13787,   318,   407,   257],
        [  835,   510,    13,  1649]])


## Step 3b: Input embeddings

The input to the transformer is built from two parts that are **added** together:

1. **Token embedding** (`wte`): *what* the token is.
2. **Positional embedding** (`wpe`): *where* the token sits in the sequence.
   Without it, the model could not tell `"dog bites man"` from `"man bites dog"`.

```
input_embedding = token_embedding(token_ids) + position_embedding(positions)
```

**Design decision:** we reuse GPT-2's pretrained embedding tables and keep them **frozen**.
Consequences to keep in mind:
- Our model's hidden size must be **768** (GPT-2's embedding size).
- The max context length is **1024** (the number of positions in GPT-2's `wpe`).
- These tables hold about 38M parameters, which will be a big share of a small model.

In [18]:
# Reuse the GPT-2 model loaded earlier, but we only need its two embedding tables
token_embedding_table = gpt2.wte         # (50257, 768): one vector per token ID
position_embedding_table = gpt2.wpe      # (1024, 768): one vector per position

# Freeze both so they are not updated during our training
for p in token_embedding_table.parameters():
    p.requires_grad = False
for p in position_embedding_table.parameters():
    p.requires_grad = False

# Token embeddings: (batch_size, context_length, 768)
token_embeddings = token_embedding_table(input_batch)

# Position embeddings: positions are 0, 1, ..., context_length - 1
# unsqueeze(0) makes the shape (1, context_length) so it can broadcast across the batch
positions = torch.arange(context_length, device=input_batch.device).unsqueeze(0)
position_embeddings = position_embedding_table(positions)    # (1, context_length, 768)

# Add them (the position embeddings are broadcast over the batch dimension)
input_embeddings = token_embeddings + position_embeddings

print(input_embeddings.shape)            # torch.Size([4, 4, 768])
print(input_embeddings)

torch.Size([4, 4, 768])
tensor([[[-0.0073, -0.2004,  0.0363,  ..., -0.0153, -0.0014, -0.0054],
         [ 0.1300,  0.0968,  0.1211,  ...,  0.0384, -0.1134, -0.1919],
         [ 0.1245, -0.0632,  0.1694,  ...,  0.0671,  0.0945, -0.0820],
         [-0.0232, -0.1707,  0.2156,  ...,  0.0175,  0.0687, -0.0302]],

        [[-0.0581, -0.1924,  0.0461,  ..., -0.0908,  0.0952,  0.0074],
         [ 0.1659, -0.0934,  0.0939,  ..., -0.0268,  0.2096, -0.0020],
         [ 0.0509, -0.0961,  0.0828,  ..., -0.0538,  0.0689,  0.0749],
         [-0.0016, -0.0559,  0.1588,  ...,  0.0015, -0.0352, -0.0418]],

        [[-0.0179, -0.1109,  0.0586,  ..., -0.1630,  0.1629,  0.0085],
         [ 0.1226, -0.0543,  0.0078,  ...,  0.0346, -0.0744, -0.0320],
         [-0.0055, -0.0747,  0.1101,  ...,  0.1342, -0.0187, -0.0468],
         [-0.0140, -0.0438,  0.1611,  ...,  0.0361, -0.0522,  0.1524]],

        [[-0.0600, -0.2053,  0.0647,  ..., -0.1227,  0.0945,  0.0346],
         [-0.1576, -0.1468, -0.0475,  ..., -0.1

## Step 4: LayerNorm (next)

Layer normalization rescales each token's vector to have mean 0 and variance 1, then applies a
learnable scale and shift. It keeps activations stable so deep networks train well.

GPT-2 uses **pre-LN**: LayerNorm is applied *before* attention and before the feed-forward network.

In [19]:
# LayerNorm implementation
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, embedding_dim = 768):
        super().__init__()

        self.layer_norm = nn.LayerNorm(embedding_dim)

    def forward(self, input_tensor):
        return self.layer_norm(input_tensor)


layerNorm = LayerNorm()
output = layerNorm(input_embeddings)
print(input_embeddings)
print(output.shape)
print(output)



tensor([[[-0.0073, -0.2004,  0.0363,  ..., -0.0153, -0.0014, -0.0054],
         [ 0.1300,  0.0968,  0.1211,  ...,  0.0384, -0.1134, -0.1919],
         [ 0.1245, -0.0632,  0.1694,  ...,  0.0671,  0.0945, -0.0820],
         [-0.0232, -0.1707,  0.2156,  ...,  0.0175,  0.0687, -0.0302]],

        [[-0.0581, -0.1924,  0.0461,  ..., -0.0908,  0.0952,  0.0074],
         [ 0.1659, -0.0934,  0.0939,  ..., -0.0268,  0.2096, -0.0020],
         [ 0.0509, -0.0961,  0.0828,  ..., -0.0538,  0.0689,  0.0749],
         [-0.0016, -0.0559,  0.1588,  ...,  0.0015, -0.0352, -0.0418]],

        [[-0.0179, -0.1109,  0.0586,  ..., -0.1630,  0.1629,  0.0085],
         [ 0.1226, -0.0543,  0.0078,  ...,  0.0346, -0.0744, -0.0320],
         [-0.0055, -0.0747,  0.1101,  ...,  0.1342, -0.0187, -0.0468],
         [-0.0140, -0.0438,  0.1611,  ...,  0.0361, -0.0522,  0.1524]],

        [[-0.0600, -0.2053,  0.0647,  ..., -0.1227,  0.0945,  0.0346],
         [-0.1576, -0.1468, -0.0475,  ..., -0.1734, -0.0483, -0.0857],


## Step 5: Masked multi-head causal attention (next)

Each token looks at the tokens before it (never the ones after it) to build a context-aware representation.

In [20]:
# TODO: masked multi-head causal attention

import torch
import torch.nn as nn
import numpy

batch = 4
tokens = 4
dimensions = 768
heads = 12
head_dimension = dimensions//heads

class MaskedMultiHeadCausalAttention(nn.Module):
    def __init__(self, emb_dim= dimensions, num_heads = heads):
        super().__init__()
        #store the model configuration
        self.embedding_dimension = emb_dim
        self.attention_heads = num_heads

        # build q_proj, k_proj, v_proj
        self.q_proj = nn.Linear(emb_dim,emb_dim)
        self.k_proj = nn.Linear(emb_dim,emb_dim)
        self.v_proj = nn.Linear(emb_dim,emb_dim)

        #head dimension of the model
        self.head_dim = self.embedding_dimension // self.attention_heads

        #output projection
        self.o_proj = nn.Linear(emb_dim, emb_dim)

    def forward(self, x = input_embeddings):
        B,T,D = x.shape # x_shape, sequence_length, emb_dim

        # build the query, key and value matrices
        # Q = X@Q + b(q),  K = X@K + b(k),   V = X@V + b(v)
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        #split the last emb_dimension into 12 attention heads and head_dimension 768 -> 12,64
        Q = Q.reshape(B, T, self.attention_heads, self.head_dim)
        K = K.reshape(B, T, self.attention_heads, self.head_dim)
        V = V.reshape(B, T, self.attention_heads, self.head_dim)

        #change the shape of the Q,K,V matrices into (batch, attention_heads, sequence_length, head_dim)
        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        # Now for the fun calculation part calulate the Queries x Keys(transpose)
        # Query dimension (4,12,4,64) , Key(T) dimension (4,12,64,4) , attention_score dimension (4,12,4,4)
        
        attention_scores = Q @ K.transpose(-2,-1)

        #scale the attention_scores
        """ S = Q*K(T)/sqrt(head_dim) """
        attention_scores = attention_scores // (self.head_dim ** 0.5) # (4,12,4,4)


        #Create the Causal Mask
        # lower triangular matrix with upper values as -ve infinity
        mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool),
            diagonal=1
        )

        attention_scores = attention_scores.masked_fill(mask, float("-inf"))

        # Now let's apply softmax to convert the attention scores to attention_weights by applying softmax along the rows
        attention_weights = torch.softmax(attention_scores, dim = -1) # (4,12,4,4)

        # Now let's multiply the attention_weights by the value matrix
        # attention_weights (4,12,4,4) , V(4,12,4,64) -> output (4,12,4,64)
        output = attention_weights @ V


        # merge the 12 heads now
        # output (4,12,4,64) -> transpose -> output(4,4,12,64)
        output = output.transpose(1,2) # (4,4,12,64)

        #merge the 12 attention heads
        output = output.contiguous().view(B, T, D)

        #Apply output projection
        # O@o_proj + b(o)            
        # (4,4,768) @ (768,768) -> (4,4,768)
        final_output = self.o_proj(output) # (4,4,768)

        return final_output 


In [21]:
#create a class of CausalMultiheadAttention
sample = MaskedMultiHeadCausalAttention()
#run the input_embeddings
output = sample(input_embeddings)

print(input_embeddings.shape)
print(output.shape)

torch.Size([4, 4, 768])
torch.Size([4, 4, 768])
